In [1]:
import torch
import torchvision
import snntorch

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("snnTorch:", snntorch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA used by PyTorch:", torch.version.cuda)

PyTorch: 2.13.0+cu130
Torchvision: 0.28.0+cu130
snnTorch: 1.0.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA used by PyTorch: 13.0


In [2]:
import torchvision
import torchvision.transforms as transforms

dataset_path = r"C:\SNN_project\cifar-10-python"

train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=True,
    download=False,
    transform=transforms.ToTensor()
)

test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=False,
    download=False,
    transform=transforms.ToTensor()
)

print("Training images:", len(train_dataset))
print("Testing images :", len(test_dataset))

Training images: 50000
Testing images : 10000


In [3]:
import torch

# ==========================================
# Convert ALL training data
# ==========================================

train_images = torch.tensor(
    train_dataset.data,
    dtype=torch.float32
) / 255.0

train_labels = torch.tensor(
    train_dataset.targets,
    dtype=torch.long
)

# ==========================================
# Convert ALL testing data
# ==========================================

test_images = torch.tensor(
    test_dataset.data,
    dtype=torch.float32
) / 255.0

test_labels = torch.tensor(
    test_dataset.targets,
    dtype=torch.long
)

# ==========================================
# Change format:
# [N, 32, 32, 3] → [N, 3, 32, 32]
# ==========================================

train_images = train_images.permute(0, 3, 1, 2)
test_images = test_images.permute(0, 3, 1, 2)

print("Training images:", train_images.shape)
print("Training labels:", train_labels.shape)

print("Testing images :", test_images.shape)
print("Testing labels :", test_labels.shape)

Training images: torch.Size([50000, 3, 32, 32])
Training labels: torch.Size([50000])
Testing images : torch.Size([10000, 3, 32, 32])
Testing labels : torch.Size([10000])


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Using device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU memory: 7.96 GB


In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

import snntorch as snn
from snntorch import surrogate
from snntorch import utils

# ==========================================
# V4 PARAMETERS
# ==========================================

num_steps_v4 = 32
batch_size_v4 = 16

num_epochs_v4 = 30

learning_rate_v4 = 1e-3
weight_decay_v4 = 1e-4
beta_v4 = 0.85

spike_grad_v4 = surrogate.fast_sigmoid(
    slope=25
)

print("Timesteps :", num_steps_v4)
print("Batch size:", batch_size_v4)
print("Epochs    :", num_epochs_v4)
print("Learning rate:", learning_rate_v4)
print("Beta:", beta_v4)

Timesteps : 32
Batch size: 16
Epochs    : 30
Learning rate: 0.001
Beta: 0.85


In [6]:
def augment_batch(images):

    images = images.clone()

    flip_mask = (
        torch.rand(
            images.size(0),
            device=images.device
        ) < 0.5
    )

    if flip_mask.any():

        images[flip_mask] = torch.flip(
            images[flip_mask],
            dims=[3]
        )

    return images
    # data augmentation

In [3]:
#residual block
class ResidualSNNBlock(nn.Module):

    def __init__(
        self,
        channels,
        beta,
        spike_grad
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.lif1 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=True
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(channels)

        self.lif2 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=True
        )

    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.lif1(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.lif2(out)

        out = out + identity

        return out

In [4]:
#complete V4 SNN_Model
class CIFAR10SNN_V4(nn.Module):

    def __init__(self):

        super().__init__()

        # ==========================================
        # Input: 3 × 32 × 32
        # ==========================================

        self.conv0 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.lif0 = snn.Leaky(
            beta=beta_v4,
            spike_grad=spike_grad_v4,
            init_hidden=True
        )

        self.pool0 = nn.MaxPool2d(2)

        # 32 × 32 → 16 × 16

        # ==========================================
        # Residual block 1
        # ==========================================

        self.res1 = ResidualSNNBlock(
            64,
            beta_v4,
            spike_grad_v4
        )

        # ==========================================
        # 64 → 128
        # ==========================================

        self.conv1 = nn.Conv2d(
            64,
            128,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(128)

        self.lif1 = snn.Leaky(
            beta=beta_v4,
            spike_grad=spike_grad_v4,
            init_hidden=True
        )

        self.pool1 = nn.MaxPool2d(2)

        # 16 × 16 → 8 × 8

        # ==========================================
        # Residual block 2
        # ==========================================

        self.res2 = ResidualSNNBlock(
            128,
            beta_v4,
            spike_grad_v4
        )

        # ==========================================
        # 128 → 256
        # ==========================================

        self.conv2 = nn.Conv2d(
            128,
            256,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(256)

        self.lif2 = snn.Leaky(
            beta=beta_v4,
            spike_grad=spike_grad_v4,
            init_hidden=True
        )

        self.pool2 = nn.MaxPool2d(2)

        # 8 × 8 → 4 × 4

        # ==========================================
        # Residual block 3
        # ==========================================

        self.res3 = ResidualSNNBlock(
            256,
            beta_v4,
            spike_grad_v4
        )

        # ==========================================
        # Classifier
        # ==========================================

        self.fc = nn.Linear(
            256 * 4 * 4,
            10
        )

        self.lif_out = snn.Leaky(
            beta=beta_v4,
            spike_grad=spike_grad_v4,
            init_hidden=True,
            output=True
        )

    def forward(self, x):

        # Input block
        x = self.conv0(x)
        x = self.bn0(x)
        x = self.lif0(x)
        x = self.pool0(x)

        # Residual block 1
        x = self.res1(x)

        # 64 → 128
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.lif1(x)
        x = self.pool1(x)

        # Residual block 2
        x = self.res2(x)

        # 128 → 256
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.lif2(x)
        x = self.pool2(x)

        # Residual block 3
        x = self.res3(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Output
        x = self.fc(x)

        spk_out, mem_out = self.lif_out(x)

        return spk_out, mem_out

In [9]:
#create V4 on the RTX5060
net_v4 = CIFAR10SNN_V4().to(device)

print(net_v4)

total_params = sum(
    p.numel()
    for p in net_v4.parameters()
)

print()
print("Total parameters:", total_params)
print(
    "Parameters in millions:",
    total_params / 1e6
)

CIFAR10SNN_V4(
  (conv0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (lif0): Leaky()
  (pool0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (res1): ResidualSNNBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (lif1): Leaky()
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (lif2): Leaky()
  )
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (lif1): Leaky()
  (pool1): MaxPool2d(kerne

In [10]:
#checking GPU memory before testing
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        3
    ),
    "GB"
)

print(
    "GPU reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        3
    ),
    "GB"
)

GPU allocated: 0.007 GB
GPU reserved: 0.021 GB


In [11]:
#this is only a memory test not training
test_batch = train_images[:batch_size_v4].to(device)

utils.reset(net_v4)

output_spikes = []
output_membranes = []

with torch.no_grad():

    for step in range(num_steps_v4):

        input_spikes = (
            torch.rand_like(test_batch)
            < test_batch
        ).float()

        spk_out, mem_out = net_v4(
            input_spikes
        )

        output_spikes.append(spk_out)
        output_membranes.append(mem_out)

output_spikes = torch.stack(
    output_spikes
)

output_membranes = torch.stack(
    output_membranes
)

print("Output spikes shape:",
      output_spikes.shape)

print("Output membrane shape:",
      output_membranes.shape)

print(
    "GPU allocated after test:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        3
    ),
    "GB"
)

Output spikes shape: torch.Size([32, 16, 10])
Output membrane shape: torch.Size([32, 16, 10])
GPU allocated after test: 0.06 GB


In [14]:
# ==========================================
# REAL TRAINING MEMORY TEST
# ==========================================

import torch

# Make sure the model is in training mode
net_v4.train()

# Create optimizer and loss
optimizer_v4 = torch.optim.AdamW(
    net_v4.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

loss_fn_v4 = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

# Use only one batch for this test
images = train_images[:batch_size_v4].to(device)
labels = train_labels[:batch_size_v4].to(device)

# Reset neuron states
utils.reset(net_v4)

output_membranes = []

# Run the 32 timesteps
for step in range(num_steps_v4):

    input_spikes = (
        torch.rand_like(images) < images
    ).float()

    spk_out, mem_out = net_v4(
        input_spikes
    )

    output_membranes.append(mem_out)

# Stack timestep outputs
output_membranes = torch.stack(
    output_membranes
)

# Final membrane
final_membrane = output_membranes[-1]

# Calculate loss
loss = loss_fn_v4(
    final_membrane,
    labels
)

# Backpropagation
optimizer_v4.zero_grad()
loss.backward()

# Gradient clipping
torch.nn.utils.clip_grad_norm_(
    net_v4.parameters(),
    max_norm=1.0
)

optimizer_v4.step()

print("Training batch completed successfully.")
print("Loss:", loss.item())

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        3
    ),
    "GB"
)

print(
    "GPU reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        3
    ),
    "GB"
)

Training batch completed successfully.
Loss: 2.461383104324341
GPU allocated: 0.114 GB
GPU reserved: 1.277 GB


In [15]:
del images
del labels
del input_spikes
del output_membranes
del final_membrane
del spk_out
del mem_out

torch.cuda.empty_cache()

print("GPU memory cleaned.")

GPU memory cleaned.


In [16]:
#creating V4 optimizer and scheduler
import math
import torch
import torch.nn as nn

num_epochs_v4 = 30
warmup_epochs = 3

optimizer_v4 = torch.optim.AdamW(
    net_v4.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

loss_fn_v4 = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

def lr_lambda(epoch):

    # Warm-up
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs

    # Cosine decay
    progress = (
        epoch - warmup_epochs
    ) / (
        num_epochs_v4 - warmup_epochs
    )

    return 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

scheduler_v4 = torch.optim.lr_scheduler.LambdaLR(
    optimizer_v4,
    lr_lambda
)

print("V4 optimizer ready.")
print("Initial learning rate:",
      optimizer_v4.param_groups[0]["lr"])

V4 optimizer ready.
Initial learning rate: 0.0003333333333333333


In [17]:
def train_one_epoch_v4(
    net,
    train_images,
    train_labels,
    batch_size,
    num_steps,
    optimizer,
    loss_fn
):

    net.train()

    num_samples = train_images.shape[0]

    # Shuffle ALL 50,000 images
    indices = torch.randperm(num_samples)

    total_loss = 0.0
    correct = 0
    total = 0

    num_batches = math.ceil(
        num_samples / batch_size
    )

    for batch_number, start in enumerate(
        range(0, num_samples, batch_size),
        start=1
    ):

        end = min(
            start + batch_size,
            num_samples
        )

        batch_indices = indices[start:end]

        images = train_images[
            batch_indices
        ].to(device)

        labels = train_labels[
            batch_indices
        ].to(device)

        # -------------------------------------
        # Random horizontal flip
        # -------------------------------------

        images = augment_batch(images)

        # -------------------------------------
        # Reset all LIF states
        # -------------------------------------

        utils.reset(net)

        output_membranes = []

        # -------------------------------------
        # 32 SNN timesteps
        # -------------------------------------

        for step in range(num_steps):

            input_spikes = (
                torch.rand_like(images)
                < images
            ).float()

            spk_out, mem_out = net(
                input_spikes
            )

            output_membranes.append(
                mem_out
            )

        # -------------------------------------
        # Stack time
        # -------------------------------------

        output_membranes = torch.stack(
            output_membranes
        )

        # -------------------------------------
        # Final membrane classification
        # -------------------------------------

        final_membrane = (
            output_membranes[-1]
        )

        loss = loss_fn(
            final_membrane,
            labels
        )

        # -------------------------------------
        # Backpropagation
        # -------------------------------------

        optimizer.zero_grad()

        loss.backward()

        # Prevent excessively large gradients
        torch.nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # -------------------------------------
        # Statistics
        # -------------------------------------

        predictions = final_membrane.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        total_loss += loss.item()

        # -------------------------------------
        # Progress
        # -------------------------------------

        if (
            batch_number == 1
            or batch_number % 50 == 0
            or batch_number == num_batches
        ):

            print(
                f"Batch {batch_number}/{num_batches} "
                f"| Images {end}/{num_samples}"
            )

    average_loss = (
        total_loss / num_batches
    )

    accuracy = (
        100.0 * correct / total
    )

    return average_loss, accuracy

In [18]:
#checkpoint function
def save_checkpoint(
    filename,
    epoch,
    net,
    optimizer,
    scheduler,
    train_loss,
    train_accuracy
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "train_accuracy": train_accuracy
        },
        filename
    )

In [19]:
#start real training
train_losses_v4 = []
train_accuracies_v4 = []
learning_rates_v4 = []

best_accuracy = 0.0

for epoch in range(num_epochs_v4):

    current_lr = (
        optimizer_v4.param_groups[0]["lr"]
    )

    print()
    print("=" * 65)
    print(
        f"V4 EPOCH {epoch + 1}/{num_epochs_v4}"
    )
    print("=" * 65)

    # Reset peak-memory statistics
    torch.cuda.reset_peak_memory_stats()

    loss_v4, accuracy_v4 = train_one_epoch_v4(
        net_v4,
        train_images,
        train_labels,
        batch_size_v4,
        num_steps_v4,
        optimizer_v4,
        loss_fn_v4
    )

    train_losses_v4.append(
        loss_v4
    )

    train_accuracies_v4.append(
        accuracy_v4
    )

    learning_rates_v4.append(
        current_lr
    )

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print()
    print(
        f"Learning rate: {current_lr:.8f}"
    )

    print(
        f"Training loss: {loss_v4:.4f}"
    )

    print(
        f"Training accuracy: {accuracy_v4:.2f}%"
    )

    print(
        f"Peak GPU memory: {peak_memory:.2f} GB"
    )

    # -----------------------------------------
    # Save latest checkpoint
    # -----------------------------------------

    save_checkpoint(
        "latest_v4_checkpoint.pth",
        epoch + 1,
        net_v4,
        optimizer_v4,
        scheduler_v4,
        loss_v4,
        accuracy_v4
    )

    # -----------------------------------------
    # Save best training model
    # -----------------------------------------

    if accuracy_v4 > best_accuracy:

        best_accuracy = accuracy_v4

        torch.save(
            net_v4.state_dict(),
            "best_v4_model.pth"
        )

        print(
            f"New best training accuracy: "
            f"{best_accuracy:.2f}%"
        )

    # -----------------------------------------
    # Scheduler
    # -----------------------------------------

    scheduler_v4.step()

    print(
        f"Checkpoint saved."
    )


V4 EPOCH 1/30
Batch 1/3125 | Images 16/50000
Batch 50/3125 | Images 800/50000
Batch 100/3125 | Images 1600/50000
Batch 150/3125 | Images 2400/50000
Batch 200/3125 | Images 3200/50000
Batch 250/3125 | Images 4000/50000
Batch 300/3125 | Images 4800/50000
Batch 350/3125 | Images 5600/50000
Batch 400/3125 | Images 6400/50000
Batch 450/3125 | Images 7200/50000
Batch 500/3125 | Images 8000/50000
Batch 550/3125 | Images 8800/50000
Batch 600/3125 | Images 9600/50000
Batch 650/3125 | Images 10400/50000
Batch 700/3125 | Images 11200/50000
Batch 750/3125 | Images 12000/50000
Batch 800/3125 | Images 12800/50000
Batch 850/3125 | Images 13600/50000
Batch 900/3125 | Images 14400/50000
Batch 950/3125 | Images 15200/50000
Batch 1000/3125 | Images 16000/50000


KeyboardInterrupt: 

In [20]:
torch.cuda.empty_cache()

batch_size_v4 = 64

print("New batch size:", batch_size_v4)

New batch size: 64


In [21]:
# ==========================================
# V4 BATCH-64 MEMORY/SPEED TEST
# ==========================================

import time
import torch

net_v4.train()

images = train_images[:batch_size_v4].to(device)
labels = train_labels[:batch_size_v4].to(device)

utils.reset(net_v4)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.time()

output_membranes = []

for step in range(num_steps_v4):

    input_spikes = (
        torch.rand_like(images) < images
    ).float()

    spk_out, mem_out = net_v4(
        input_spikes
    )

    output_membranes.append(mem_out)

output_membranes = torch.stack(
    output_membranes
)

final_membrane = output_membranes[-1]

loss = loss_fn_v4(
    final_membrane,
    labels
)

optimizer_v4.zero_grad()

loss.backward()

torch.nn.utils.clip_grad_norm_(
    net_v4.parameters(),
    max_norm=1.0
)

optimizer_v4.step()

torch.cuda.synchronize()

elapsed = time.time() - start_time

peak_memory = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("====================================")
print("BATCH 64 TEST")
print("====================================")
print("Batch size:", batch_size_v4)
print("Loss:", loss.item())
print(f"Time: {elapsed:.2f} seconds")
print(f"Peak GPU memory: {peak_memory:.2f} GB")

BATCH 64 TEST
Batch size: 64
Loss: 2.1961889266967773
Time: 0.80 seconds
Peak GPU memory: 4.81 GB


In [22]:
# ==========================================
# RESET V4 FOR OFFICIAL TRAINING
# ==========================================

torch.cuda.empty_cache()

net_v4 = CIFAR10SNN_V4().to(device)

print("Fresh V4 model created.")

total_params = sum(
    p.numel()
    for p in net_v4.parameters()
)

print(
    "Total parameters:",
    total_params
)
print(
    "Parameters (millions):",
    total_params / 1e6
)

Fresh V4 model created.
Total parameters: 1962314
Parameters (millions): 1.962314


In [23]:
# ==========================================
# FINAL V4 TRAINING CONFIGURATION
# ==========================================

num_steps_v4 = 32
batch_size_v4 = 64
num_epochs_v4 = 30

warmup_epochs = 3

learning_rate_v4 = 1e-3
weight_decay_v4 = 1e-4

optimizer_v4 = torch.optim.AdamW(
    net_v4.parameters(),
    lr=learning_rate_v4,
    weight_decay=weight_decay_v4
)

loss_fn_v4 = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

def lr_lambda(epoch):

    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs

    progress = (
        epoch - warmup_epochs
    ) / (
        num_epochs_v4 - warmup_epochs
    )

    return 0.5 * (
        1.0 + math.cos(
            math.pi * progress
        )
    )

scheduler_v4 = torch.optim.lr_scheduler.LambdaLR(
    optimizer_v4,
    lr_lambda
)

print("Timesteps:", num_steps_v4)
print("Batch size:", batch_size_v4)
print("Epochs:", num_epochs_v4)
print("Initial LR:", learning_rate_v4)

Timesteps: 32
Batch size: 64
Epochs: 30
Initial LR: 0.001


In [24]:
def train_one_epoch_v4(
    net,
    train_images,
    train_labels,
    batch_size,
    num_steps,
    optimizer,
    loss_fn
):

    net.train()

    num_samples = train_images.shape[0]

    # Shuffle all 50,000 samples
    indices = torch.randperm(num_samples)

    total_loss = 0.0
    correct = 0
    total = 0

    num_batches = math.ceil(
        num_samples / batch_size
    )

    for batch_number, start in enumerate(
        range(0, num_samples, batch_size),
        start=1
    ):

        end = min(
            start + batch_size,
            num_samples
        )

        batch_indices = indices[start:end]

        images = train_images[
            batch_indices
        ].to(device)

        labels = train_labels[
            batch_indices
        ].to(device)

        # Data augmentation
        images = augment_batch(images)

        # Reset all LIF states
        utils.reset(net)

        output_membranes = []

        # ======================================
        # 32 timesteps
        # ======================================

        for step in range(num_steps):

            input_spikes = (
                torch.rand_like(images)
                < images
            ).float()

            spk_out, mem_out = net(
                input_spikes
            )

            output_membranes.append(
                mem_out
            )

        output_membranes = torch.stack(
            output_membranes
        )

        # Final membrane
        final_membrane = (
            output_membranes[-1]
        )

        loss = loss_fn(
            final_membrane,
            labels
        )

        # Backpropagation
        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # Accuracy
        predictions = final_membrane.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        total_loss += loss.item()

        # Progress
        if (
            batch_number == 1
            or
            batch_number % 100 == 0
            or
            batch_number == num_batches
        ):

            print(
                f"Batch {batch_number}/{num_batches} "
                f"| Images {end}/{num_samples}"
            )

    average_loss = (
        total_loss / num_batches
    )

    accuracy = (
        100.0 * correct / total
    )

    return average_loss, accuracy

In [25]:
#checkpoint function
def save_checkpoint(
    filename,
    epoch,
    net,
    optimizer,
    scheduler,
    train_loss,
    train_accuracy
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "train_accuracy": train_accuracy
        },
        filename
    )

In [ ]:
#restart official V4 training
train_losses_v4 = []
train_accuracies_v4 = []
learning_rates_v4 = []

best_accuracy = 0.0

for epoch in range(num_epochs_v4):

    current_lr = (
        optimizer_v4.param_groups[0]["lr"]
    )

    print()
    print("=" * 65)
    print(
        f"V4 EPOCH {epoch + 1}/{num_epochs_v4}"
    )
    print("=" * 65)

    torch.cuda.reset_peak_memory_stats()

    loss_v4, accuracy_v4 = train_one_epoch_v4(
        net_v4,
        train_images,
        train_labels,
        batch_size_v4,
        num_steps_v4,
        optimizer_v4,
        loss_fn_v4
    )

    train_losses_v4.append(
        loss_v4
    )

    train_accuracies_v4.append(
        accuracy_v4
    )

    learning_rates_v4.append(
        current_lr
    )

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print()
    print(f"Learning rate: {current_lr:.8f}")
    print(f"Training loss: {loss_v4:.4f}")
    print(f"Training accuracy: {accuracy_v4:.2f}%")
    print(f"Peak GPU memory: {peak_memory:.2f} GB")

    # Save latest checkpoint
    save_checkpoint(
        "latest_v4_checkpoint.pth",
        epoch + 1,
        net_v4,
        optimizer_v4,
        scheduler_v4,
        loss_v4,
        accuracy_v4
    )

    # Save best training model
    if accuracy_v4 > best_accuracy:

        best_accuracy = accuracy_v4

        torch.save(
            net_v4.state_dict(),
            "best_v4_model.pth"
        )

        print(
            f"New best training accuracy: "
            f"{best_accuracy:.2f}%"
        )

    scheduler_v4.step()

    print("Checkpoint saved.")

In [27]:
import os

print("Checkpoint exists:",
      os.path.exists("latest_v4_checkpoint.pth"))

if os.path.exists("latest_v4_checkpoint.pth"):
    print(
        "Checkpoint size:",
        round(
            os.path.getsize("latest_v4_checkpoint.pth")
            / (1024**2),
            2
        ),
        "MB"
    )

Checkpoint exists: True
Checkpoint size: 22.52 MB


In [28]:
checkpoint = torch.load(
    "latest_v4_checkpoint.pth",
    map_location=device
)

print("Last saved epoch:",
      checkpoint["epoch"])

print("Saved training loss:",
      checkpoint["train_loss"])

print("Saved training accuracy:",
      checkpoint["train_accuracy"])

Last saved epoch: 13
Saved training loss: 1.1022091298304555
Saved training accuracy: 74.096


In [29]:
# ==========================================
# RESTORE V4 FROM EPOCH 13
# ==========================================

checkpoint = torch.load(
    "latest_v4_checkpoint.pth",
    map_location=device
)

net_v4.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer_v4.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler_v4.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

last_epoch = checkpoint["epoch"]

print("Checkpoint restored successfully.")
print("Last completed epoch:", last_epoch)
print("Saved training loss:", checkpoint["train_loss"])
print("Saved training accuracy:", checkpoint["train_accuracy"])

Checkpoint restored successfully.
Last completed epoch: 13
Saved training loss: 1.1022091298304555
Saved training accuracy: 74.096


In [30]:
# Move scheduler from epoch 13 to epoch 14
scheduler_v4.step()

print(
    "Learning rate for next epoch:",
    optimizer_v4.param_groups[0]["lr"]
)

Learning rate for next epoch: 0.0006980398830195785


In [31]:
# ==========================================
# RESUME V4 TRAINING: EPOCH 14 → 30
# ==========================================

num_epochs_v4 = 30

train_losses_v4 = []
train_accuracies_v4 = []
learning_rates_v4 = []

best_accuracy = checkpoint["train_accuracy"]

for epoch in range(last_epoch, num_epochs_v4):

    current_lr = (
        optimizer_v4.param_groups[0]["lr"]
    )

    print()
    print("=" * 65)
    print(
        f"V4 EPOCH {epoch + 1}/{num_epochs_v4}"
    )
    print("=" * 65)

    torch.cuda.reset_peak_memory_stats()

    loss_v4, accuracy_v4 = train_one_epoch_v4(
        net_v4,
        train_images,
        train_labels,
        batch_size_v4,
        num_steps_v4,
        optimizer_v4,
        loss_fn_v4
    )

    train_losses_v4.append(loss_v4)
    train_accuracies_v4.append(accuracy_v4)
    learning_rates_v4.append(current_lr)

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print()
    print(f"Learning rate: {current_lr:.8f}")
    print(f"Training loss: {loss_v4:.4f}")
    print(f"Training accuracy: {accuracy_v4:.2f}%")
    print(f"Peak GPU memory: {peak_memory:.2f} GB")

    # Save checkpoint
    save_checkpoint(
        "latest_v4_checkpoint.pth",
        epoch + 1,
        net_v4,
        optimizer_v4,
        scheduler_v4,
        loss_v4,
        accuracy_v4
    )

    # Save best model
    if accuracy_v4 > best_accuracy:

        best_accuracy = accuracy_v4

        torch.save(
            net_v4.state_dict(),
            "best_v4_model.pth"
        )

        print(
            f"New best training accuracy: "
            f"{best_accuracy:.2f}%"
        )

    # Move scheduler to next epoch
    scheduler_v4.step()

    print("Checkpoint saved.")


V4 EPOCH 14/30
Batch 1/782 | Images 64/50000
Batch 100/782 | Images 6400/50000
Batch 200/782 | Images 12800/50000
Batch 300/782 | Images 19200/50000
Batch 400/782 | Images 25600/50000
Batch 500/782 | Images 32000/50000
Batch 600/782 | Images 38400/50000
Batch 700/782 | Images 44800/50000
Batch 782/782 | Images 50000/50000

Learning rate: 0.00069804
Training loss: 1.0524
Training accuracy: 76.01%
Peak GPU memory: 9.22 GB
New best training accuracy: 76.01%
Checkpoint saved.

V4 EPOCH 15/30
Batch 1/782 | Images 64/50000
Batch 100/782 | Images 6400/50000
Batch 200/782 | Images 12800/50000
Batch 300/782 | Images 19200/50000
Batch 400/782 | Images 25600/50000
Batch 500/782 | Images 32000/50000
Batch 600/782 | Images 38400/50000
Batch 700/782 | Images 44800/50000
Batch 782/782 | Images 50000/50000

Learning rate: 0.00064340
Training loss: 1.0290
Training accuracy: 76.61%
Peak GPU memory: 9.21 GB
New best training accuracy: 76.61%
Checkpoint saved.

V4 EPOCH 16/30
Batch 1/782 | Images 64/5000

KeyboardInterrupt: 

In [32]:
# ==========================================
# RESTORE V4 FROM EPOCH 13
# ==========================================

checkpoint = torch.load(
    "latest_v4_checkpoint.pth",
    map_location=device
)

net_v4.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer_v4.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler_v4.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

last_epoch = checkpoint["epoch"]

print("Checkpoint restored successfully.")
print("Last completed epoch:", last_epoch)
print("Saved training loss:", checkpoint["train_loss"])
print("Saved training accuracy:", checkpoint["train_accuracy"])

Checkpoint restored successfully.
Last completed epoch: 17
Saved training loss: 0.9580137709827374
Saved training accuracy: 79.106


In [35]:
print("Allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
print("Reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")
print("Peak allocated:",
      round(torch.cuda.max_memory_allocated() / 1024**3, 3), "GB")

Allocated: 7.005 GB
Reserved: 9.588 GB
Peak allocated: 9.209 GB


In [5]:
# Recreate the V4 model without training
torch.cuda.empty_cache()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

net_v4 = CIFAR10SNN_V4().to(device)

print("Fresh V4 model created.")
print("Device:", device)
print("Total parameters:", sum(
    p.numel() for p in net_v4.parameters()
))

Fresh V4 model created.
Device: cuda
Total parameters: 1962314


In [6]:
from pathlib import Path

checkpoint_path = Path(
    r"C:\SNN_project\latest_v4_checkpoint.pth"
)

print("Checkpoint exists:", checkpoint_path.exists())

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

net_v4.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Checkpoint restored successfully!")
print("Saved epoch:", checkpoint["epoch"])
print("Saved training loss:", checkpoint["train_loss"])
print("Saved training accuracy:", checkpoint["train_accuracy"])

Checkpoint exists: True
Checkpoint restored successfully!
Saved epoch: 30
Saved training loss: 0.6886073584291161
Saved training accuracy: 88.2


In [7]:
net_v4.eval()

print("Model is in evaluation mode:", not net_v4.training)
print("Model device:", next(net_v4.parameters()).device)
print("Ready for inference:", True)

Model is in evaluation mode: True
Model device: cuda:0
Ready for inference: True
